# DEDAL — Alineamiento de secuencias de proteínas

**DEDAL** (*Deep Embedding and Differentiable ALignment*) es un modelo de Google Research que alinea pares de secuencias de proteínas. La idea central es esta:

> El Smith-Waterman clásico usa una matriz de sustitución **fija** (BLOSUM62, PAM250...), igual para todas las proteínas. DEDAL en cambio **aprende** los parámetros de Smith-Waterman a partir del contexto de cada par concreto de secuencias, usando un transformer.

Es decir: en lugar de preguntar *"¿cuánto vale en general cambiar una D por una E?"*, pregunta *"¿cuánto vale cambiar **esta** D por **esta** E, dadas las secuencias en las que aparecen?"*. Eso lo hace mucho mejor en el **twilight zone**: pares de proteínas con baja identidad de secuencia (<25 %) pero que aun así son homólogas.

El modelo hace dos cosas a la vez:
1. **Alineamiento** — qué posición de A corresponde a qué posición de B.
2. **Detección de homología** — si las dos proteínas comparten un ancestro evolutivo.

En este notebook recorremos el pipeline completo paso a paso. Al final está el atajo de una sola línea.

---
## 0. Requisitos

Este repo es una extracción del paquete `dedal` de `google-research`, con el contenido en la raíz. Para que los imports internos (`from dedal import ...`) resuelvan hay que instalarlo en modo editable **una vez**:

```bash
python3 -m venv venv && source venv/bin/activate
pip install --upgrade pip     # los editable installs necesitan pip >= 21.3
pip install -e .
```

Y el kernel de Jupyter debe apuntar a ese venv. Si ves `ModuleNotFoundError: No module named 'dedal'`, es que falta ese paso o el kernel es otro.

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub

# Absolute import: works from any working directory once `pip install -e .`
# has been run. A bare `import infer` would only work from the repo root.
from dedal import infer

---
## 1. Cargar el modelo preentrenado

El modelo vive en TensorFlow Hub. La primera ejecución lo **descarga** (unos cientos de MB) y lo cachea en disco; las siguientes son instantáneas.

Para cambiar dónde se cachea: variable de entorno `TFHUB_CACHE_DIR`.

In [2]:
dedal_model = hub.load('https://tfhub.dev/google/dedal/3')

---
## 2. Las secuencias a comparar

Cada letra es un aminoácido en código de una letra (`S`=serina, `V`=valina, `C`=cisteína...).

Estas dos son las de la Figura 3 del paper: proteínas de **gorila** y de **pato real**. Son homólogas, pero con identidad de secuencia baja — justo el caso difícil donde DEDAL debería lucirse frente a métodos clásicos.

In [3]:
protein_a = 'SVCCRDYVRYRLPLRVVKHFYWTSDSCPRPGVVLLTFRDKEICADPRVPWVKMILNKL'
protein_b = 'VKCKCSRKGPKIRFSNVRKLEIKPRYPFCVEEMIIVTLWTRVRGEQQHCLNPKRQNTVRLLKWY'

print(f'protein_a: {len(protein_a)} aminoacids')
print(f'protein_b: {len(protein_b)} aminoacids')

protein_a: 58 aminoacids
protein_b: 64 aminoacids


---
## 3. Preprocesamiento

`infer.preprocess()` convierte los dos strings en el tensor que el modelo espera. Por dentro (ver `infer.py:29`) aplica cuatro pasos:

| Paso | Qué hace |
|---|---|
| `.strip().upper()` | Normaliza: quita espacios, pasa a mayúsculas |
| `transforms.Encode` | Cada aminoácido → su ID entero, según `vocabulary.seqio_vocab` |
| `transforms.EOS` | Añade el token especial de fin de secuencia |
| `transforms.CropOrPad` | Rellena con padding (o recorta) hasta exactamente **512** posiciones |

El resultado es `tf.Tensor<int32>[2, 512]`: una fila por secuencia.

**Convención de batching (importante si procesas varios pares).** El modelo espera `[2B, 512]` para *B* pares, con las dos mitades de cada par **consecutivas**: `inputs[0]` y `inputs[1]` son el par 0, `inputs[2]` e `inputs[3]` el par 1, etc. No es "todas las izquierdas y luego todas las derechas".

**El límite de 512 es duro.** Una proteína más larga se recorta en silencio y pierdes el extremo — sin aviso. Si trabajas con secuencias largas, compruébalo tú.

In [4]:
inputs = infer.preprocess(protein_a, protein_b)

print('shape:', inputs.shape, '| dtype:', inputs.dtype)
# First 12 token IDs of sequence A, followed by the tail where padding starts.
print('first tokens of A:', inputs[0, :12].numpy())
print(f'tokens at the A/padding boundary (len={len(protein_a)}):',
      inputs[0, len(protein_a) - 2:len(protein_a) + 3].numpy())

shape: (2, 512) | dtype: <dtype: 'int32'>
first tokens of A: [20 24  9  9  6  8 23 24  6 23  6 15]
tokens at the A/padding boundary (len=58): [ 16  15 126   0   0]


---
## 4. Ejecutar el modelo (modo alineamiento)

Por defecto el modelo corre en modo *alignment* y devuelve un `dict` con cuatro entradas:

| Clave | Forma | Significado |
|---|---|---|
| `sw_scores` | `[B]` | Score de Smith-Waterman del alineamiento óptimo |
| `homology_logits` | `[B]` | Logit de "son homólogas" (**logit, no probabilidad**) |
| `paths` | `[B, 512, 512, 9]` | El alineamiento en sí, como distribución sobre transiciones |
| `sw_params` | 3 × `[B, 512, 512]` | Los parámetros SW aprendidos: scores de sustitución, gap open, gap extend |

Lo interesante es `sw_params`: **eso** es la contribución del paper. En un Smith-Waterman clásico esos tres parámetros son constantes fijadas a mano; aquí son tensores que el transformer predice para este par concreto.

La dimensión 9 de `paths` son las transiciones del autómata de 3 estados (match, gap_open, gap_extend) → 3 × 3 = 9 combinaciones estado-anterior × estado-actual.

In [5]:
align_out = dedal_model(inputs)

for key, value in align_out.items():
    if isinstance(value, (list, tuple)):
        print(f'{key:20s} tuple of {len(value)} -> {[tuple(v.shape) for v in value]}')
    else:
        print(f'{key:20s} {tuple(value.shape)}')

sw_scores            (1,)
sw_params            tuple of 3 -> [(1, 512, 512), (1, 512, 512), (1, 512, 512)]
paths                (1, 512, 512, 9)
homology_logits      (1,)


---
## 5. Embeddings (modo alternativo)

Con `embeddings_only=True` el modelo se salta la capa de alineamiento y devuelve solo las representaciones del encoder: `[2B, 512, 768]` — un vector de 768 dimensiones **por cada posición** de cada secuencia.

Son embeddings *contextuales*: la misma letra en dos posiciones distintas produce vectores distintos, porque el transformer mira toda la secuencia. Sirven para tareas propias (clustering, clasificación, búsqueda por similitud) sin usar la parte de alineamiento.

Este paso es independiente del anterior — no hace falta para el alineamiento, va aquí solo para mostrar la otra salida.

In [6]:
embeddings = dedal_model.call(inputs, embeddings_only=True)

print('shape:', embeddings.shape)
print('-> (2 sequences, 512 positions, 768 features per position)')

shape: (2, 512, 768)
-> (2 sequences, 512 positions, 768 features per position)


---
## 6. Postproceso: de tensores a un alineamiento legible

Hacen falta tres llamadas, y conviene saber qué hace cada una:

**`infer.expand(...)`** — Reconstruye tuplas anidadas a partir de claves aplanadas tipo `output_0`, `output_1_2` que produce un `SavedModel`. **Aquí en realidad no hace nada**: solo actúa si recibe un `dict` (`infer.py:77`), y nosotros le pasamos una lista ya armada a mano. Se queda porque es lo que trae el README oficial, pero en este camino es un no-op. Lo vemos con detalle en la sección 9.

**`infer.postprocess(output, len_a, len_b)`** — Aquí sí pasan cosas (`infer.py:44`):
- quita la dimensión de batch;
- **invierte el signo** de las penalizaciones de gap, para que se lean como números positivos;
- **recorta el padding**, pasando de 512×512 a 57×64 (las longitudes reales);
- convierte `paths` de 9 transiciones a los 3 estados (match, gap_open, gap_extend).

**`infer.Alignment(...)`** — Recorre el camino óptimo y arma las dos cadenas alineadas con sus guiones y la línea de símbolos del medio.

⚠️ Ojo con el orden de los argumentos: `postprocess` recibe las longitudes **originales** de las proteínas (`len(protein_a)`, `len(protein_b)`), en el mismo orden en que se las pasaste a `preprocess`. Si los inviertes, el recorte se hace mal y el alineamiento sale corrupto.

In [7]:
# Rebuild the nested tuple structure flattened by the SavedModel.
output = infer.expand(
    [align_out['sw_scores'], align_out['paths'], align_out['sw_params']])

# Drop batch dim, flip gap-penalty signs, crop padding, collapse to 3 states.
output = infer.postprocess(output, len(protein_a), len(protein_b))

alignment = infer.Alignment(protein_a, protein_b, *output)
print(alignment)

   0  SVC-CRDYVRYRLPLRVVKHFYW--TSDSCPRPGVV-LLTF---RDKEICADPRVPWVKMILNKL  57  
      ::| |.:.: ..:...:|::::.  .:::|:.:.:: .|..   ..:::|::|:::.::::|:::      
   0  VKCKCSRKG-PKIRFSNVRKLEIKPRYPFCVEEMIIVTLWTRVRGEQQHCLNPKRQNTVRLLKWY  63  


---
## 7. Cómo leer esa salida

El bloque tiene tres líneas: **secuencia A**, **símbolos**, **secuencia B**. Los números a los lados son los índices (base 0) donde empieza y termina el alineamiento en cada secuencia.

Los símbolos del medio salen de `Alignment._position_to_char` (`infer.py:114`):

| Símbolo | Significado |
|---|---|
| `\|` | **Identidad** — el mismo aminoácido en ambas |
| `:` | **Sustitución favorable** — distintos, pero el score de sustitución aprendido es **> 0** |
| `.` | **Sustitución desfavorable** — distintos y con score **≤ 0** |
| (espacio) | **Gap** — hay un `-` en una de las dos |

Aquí está la diferencia clave con las herramientas clásicas: ese `:` **no** viene de BLOSUM62. Viene del score que el modelo predijo *para esa pareja de posiciones concreta*. Dos `D`↔`E` en sitios distintos de la proteína pueden recibir símbolos distintos.

### Métricas

La clase expone tres propiedades. **Son conteos absolutos, no porcentajes** — un error fácil de cometer al reportarlas:

- `identity` → número de `|`
- `similarity` → `identity` + número de `:`
- `gaps` → posiciones con un guion

In [8]:
n = len(alignment)  # total aligned columns, gaps included

print(f'alignment length : {n}')
print(f'identity         : {alignment.identity:3d}  ({alignment.identity / n:6.1%})')
print(f'similarity       : {alignment.similarity:3d}  ({alignment.similarity / n:6.1%})')
print(f'gaps             : {alignment.gaps:3d}  ({alignment.gaps / n:6.1%})')
print(f'spans A[{alignment.start[0]}:{alignment.end[0]}] vs B[{alignment.start[1]}:{alignment.end[1]}]')

alignment length : 65
identity         :   8  ( 12.3%)
similarity       :  40  ( 61.5%)
gaps             :   8  ( 12.3%)
spans A[0:57] vs B[0:63]


---
## 8. Los dos scores

**`sw_scores`** — El score de Smith-Waterman del alineamiento óptimo. Está marcado como *uncorrected*: es un valor crudo, **no** un e-value ni un p-value. No es comparable entre pares de longitudes muy distintas, porque secuencias más largas tienden a acumular score. Para decidir "esto es significativo" no uses este número, usa el logit de homología.

**`homology_logits`** — La predicción de homología, en escala de logit (de −∞ a +∞). Positivo tira a homólogas, negativo a no homólogas. Para leerlo como probabilidad hay que pasarlo por una sigmoide.

In [9]:
sw_score = align_out['sw_scores'].numpy()[0]
logit = align_out['homology_logits'].numpy()[0]

# The head outputs a logit; squash it to get an interpretable probability.
probability = tf.sigmoid(logit).numpy()

print(f'Smith-Waterman score (uncorrected) : {sw_score:.3f}')
print(f'Homology logit                     : {logit:.3f}')
print(f'Homology probability               : {probability:.4f}')

Smith-Waterman score (uncorrected) : 14.427
Homology logit                     : 11.421
Homology probability               : 1.0000


### Lo que dicen estos números concretos

Junta los resultados de las dos celdas anteriores:

| Medida | Valor |
|---|---|
| Identidad | **12,3 %** (8 de 65 columnas) |
| Similitud | **61,5 %** (40 de 65) |
| Probabilidad de homología | **≈ 1,0** |

Esto es el *twilight zone* en una sola tabla. Con un 12 % de identidad, una búsqueda clásica por identidad de secuencia descartaría este par sin dudarlo — el umbral habitual ronda el 25-30 %, y por debajo de eso el ruido domina. Un BLAST probablemente ni lo reportaría.

Pero DEDAL le asigna una probabilidad de homología prácticamente de 1. La señal está en el hueco entre esos dos números: solo el 12 % son coincidencias exactas, pero el **61 %** son posiciones que el modelo puntúa como sustituciones favorables (`:`). Es decir, no encuentra los *mismos* aminoácidos — encuentra aminoácidos *intercambiables en este contexto*, que es lo que la evolución realmente conserva.

Un aviso al interpretar: una probabilidad de 1,0000 es el resultado de aplastar un logit de 11,4 con una sigmoide. No la leas como certeza calibrada; cualquier logit por encima de ~8 va a redondear a 1,0. El logit crudo conserva más matiz para comparar entre pares.

---
## 9. El atajo `infer.align()` y por qué **no** sirve aquí

`infer.py:184` ofrece una función que hace todo el pipeline de un tirón:

```python
alignment = infer.align(model, left, right)   # ⚠️ falla con el modelo de TF Hub
```

Parece justo lo que queremos, pero con el modelo de TF Hub v3 revienta:

```
ValueError: invalid literal for int() with base 10: 'params'
```

**La causa.** `align()` llama a `expand()` sobre la salida cruda del modelo. Y `expand()` espera claves aplanadas de `SavedModel` (`output_0`, `output_1_2`), de las que extrae la posición con `int(k.split('_')[1:][0])`. Pero el modelo de TF Hub devuelve claves **con nombre semántico** — `sw_scores`, `paths`, `sw_params`, `homology_logits`. Al intentar `int('params')`, explota.

Dicho de otro modo: `align()` se escribió para los checkpoints internos de entrenamiento, cuya firma aplana los nombres. La versión publicada en TF Hub tiene una firma más amable, y esa función nunca se actualizó para ella.

Por eso el README oficial hace el camino largo: al construir la lista `[sw_scores, paths, sw_params]` a mano ya estás resolviendo el problema que `expand()` intentaba resolver — de ahí que sea un no-op.

**La solución**: un envoltorio propio de cinco líneas, saltándose `expand()`.

In [10]:
def align(model, left, right, max_length=512):
    """Align two sequences with a TF Hub DEDAL model.

    Equivalent to `infer.align`, minus the `expand` call that assumes the
    flattened `output_N` keys of an internal SavedModel signature.
    """
    inputs = infer.preprocess(left, right, max_length)
    out = model(inputs)
    packed = infer.postprocess(
        [out['sw_scores'], out['paths'], out['sw_params']], len(left), len(right))
    return infer.Alignment(left, right, *packed)


quick = align(dedal_model, protein_a, protein_b)
print(quick)

print('\nsame result as the step-by-step version:',
      quick.matches == alignment.matches)

   0  SVC-CRDYVRYRLPLRVVKHFYW--TSDSCPRPGVV-LLTF---RDKEICADPRVPWVKMILNKL  57  
      ::| |.:.: ..:...:|::::.  .:::|:.:.:: .|..   ..:::|::|:::.::::|:::      
   0  VKCKCSRKG-PKIRFSNVRKLEIKPRYPFCVEEMIIVTLWTRVRGEQQHCLNPKRQNTVRLLKWY  63  

same result as the step-by-step version: True


---
## Para seguir explorando

- **Varios pares a la vez** — apila los `preprocess` respetando el orden `[A0, B0, A1, B1, ...]`. Es bastante más rápido que llamar al modelo par por par.
- **Inspeccionar `sw_params`** — `alignment.sw_params` tiene forma `[len_a, len_b, 3]`: score de sustitución, gap open y gap extend por cada celda. Visualizarlo como heatmap enseña bastante sobre qué está "viendo" el modelo.
- **Control negativo** — alinea dos proteínas sin relación y compara el logit de homología. Sin un negativo con el que contrastar, un logit positivo por sí solo no dice gran cosa.
- **Paper** — [Llinares-López et al., *Deep embedding and alignment of protein sequences*](https://www.biorxiv.org/content/10.1101/2021.11.15.468653v2)